# 02 - Train the resume NER model

Fine-tunes `distilbert-base-cased` for token classification. **Runtime -> Change runtime type -> T4 GPU.**
Trains with 3 random seeds and keeps the best, so the result is not a lucky run. Selection uses the **validation** set only.

In [ ]:
import os, sys, json, glob
if not os.path.exists("InterviewAI"):
    !git clone -q https://github.com/Ishangupta-cyber/InterviewAI.git
%cd /content/InterviewAI
!git pull -q
sys.path.insert(0, "ml/src/resume_ner")
import data

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/interview_ai_ml"   # splits, models and reports live here, so Colab resets cannot erase them
os.makedirs(OUT, exist_ok=True)

!pip -q install seqeval transformers datasets accelerate

In [ ]:
import numpy as np, torch
from datasets import Dataset
from seqeval.metrics import f1_score, precision_score, recall_score
from seqeval.scheme import IOB2
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification, set_seed)

MODEL = "distilbert-base-cased"
CHUNK, STRIDE = 150, 100      # words per window / step (kept short so word-pieces stay under 512)
SEEDS = [1, 2, 3]
EPOCHS, LR, BATCH = 10, 5e-5, 16

labels = json.load(open(f"{OUT}/labels.json"))
l2i = {l: i for i, l in enumerate(labels)}
load = lambda n: data.load_jsonl(f"{OUT}/{n}_resumes.jsonl")
train, val = data.chunk_all(load("train"), CHUNK, STRIDE), data.chunk_all(load("val"), CHUNK, STRIDE)
print("train windows:", len(train), "val windows:", len(val), "labels:", len(labels))

tok = AutoTokenizer.from_pretrained(MODEL)

def encode(batch):
    enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=512)
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        prev, lab = None, []
        for w in enc.word_ids(batch_index=i):
            lab.append(-100 if (w is None or w == prev) else l2i[tags[w]])   # label only the first word-piece
            prev = w
        all_labels.append(lab)
    enc["labels"] = all_labels
    return enc

to_ds = lambda rows: Dataset.from_list(rows).map(encode, batched=True, remove_columns=["tokens", "tags"])
train_ds, val_ds = to_ds(train), to_ds(val)

def compute_metrics(p):
    logits, gold = p
    pred = logits.argmax(-1)
    y_true, y_pred = [], []
    for pr, g in zip(pred, gold):
        keep = g != -100
        y_true.append([labels[i] for i in g[keep]]); y_pred.append([labels[i] for i in pr[keep]])
    kw = dict(mode="strict", scheme=IOB2)
    return {"precision": precision_score(y_true, y_pred, **kw), "recall": recall_score(y_true, y_pred, **kw),
            "f1": f1_score(y_true, y_pred, **kw)}

## Train (3 seeds)

In [ ]:
results, best = [], {"f1": -1}
for seed in SEEDS:
    set_seed(seed)
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=l2i)
    args = TrainingArguments(
        output_dir=f"/content/run_{seed}", eval_strategy="epoch", save_strategy="epoch",
        learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=32,
        num_train_epochs=EPOCHS, weight_decay=0.01, warmup_ratio=0.1, seed=seed,
        load_best_model_at_end=True, metric_for_best_model="f1", save_total_limit=1,
        fp16=torch.cuda.is_available(), report_to="none", logging_strategy="epoch")
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorForTokenClassification(tok), compute_metrics=compute_metrics)
    trainer.train()
    m = trainer.evaluate()
    print(f"seed {seed}: val F1 {m['eval_f1']:.4f}  P {m['eval_precision']:.4f}  R {m['eval_recall']:.4f}")
    results.append({"seed": seed, "val_f1": m["eval_f1"], "val_precision": m["eval_precision"],
                    "val_recall": m["eval_recall"]})
    if m["eval_f1"] > best["f1"]:
        best = {"f1": m["eval_f1"], "seed": seed, "history": trainer.state.log_history}
        trainer.save_model(f"{OUT}/resume_ner_model"); tok.save_pretrained(f"{OUT}/resume_ner_model")
        json.dump(labels, open(f"{OUT}/resume_ner_model/labels.json", "w"))

print("best seed:", best["seed"], "val F1:", round(best["f1"], 4))
json.dump({"seeds": results, "model": MODEL, "epochs": EPOCHS, "lr": LR, "batch": BATCH,
           "chunk": CHUNK, "stride": STRIDE}, open(f"{OUT}/train_summary.json", "w"), indent=1)

## Learning curves (put this in the report)

In [ ]:
import matplotlib.pyplot as plt
h = best["history"]
tr = [(x["epoch"], x["loss"]) for x in h if "loss" in x]
ev = [(x["epoch"], x["eval_loss"], x["eval_f1"]) for x in h if "eval_loss" in x]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(*zip(*tr), label="train loss"); ax[0].plot([e[0] for e in ev], [e[1] for e in ev], label="val loss")
ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].set_title("Loss")
ax[1].plot([e[0] for e in ev], [e[2] for e in ev]); ax[1].set_xlabel("epoch"); ax[1].set_title("Validation F1")
plt.tight_layout(); plt.savefig(f"{OUT}/learning_curves.png", dpi=150); plt.show()

**Read the curves:** if train loss keeps falling but validation loss rises, the model is overfitting. With only ~176 training
resumes that is likely, so say it in the report and mention the fix (more data, e.g. the larger dataset in notebook 01).